In [10]:




import pandas as pd
from tqdm import tqdm
import os
from PIL import Image

# Path to your CSV and output folder
csv_path = "/home/brhanu/thesis_project/results/owlvit_results.csv"
output_dir = "/home/brhanu/thesis_project/data/labels/val"

# Load CSV
df = pd.read_csv(csv_path)
print(f"✅ Loaded {len(df)} detections")
print(df.columns)

# --- Dynamic threshold based on confidence scores ---
if 'score' in df.columns:
    CONF_THRESH = df['score'].quantile(0.25)  # 25th percentile
    print(f"⚙️ Using dynamic CONF_THRESH = {CONF_THRESH:.3f}")
else:
    raise KeyError("Column 'score' not found in CSV. Check your CSV header names.")

# Filter by confidence
df = df[df['score'] >= CONF_THRESH]
print(f"➡️ After filtering: {len(df)} detections kept (confidence ≥ {CONF_THRESH:.3f})")

# Create output folder
os.makedirs(output_dir, exist_ok=True)

# --- Group detections per image and save as YOLO .txt files ---
for image_path, group in tqdm(df.groupby('image_path'), desc="Generating YOLO labels"):
    base_name = os.path.splitext(os.path.basename(image_path))[0]
    label_path = os.path.join(output_dir, f"{base_name}.txt")

    # Get actual image dimensions
    try:
        with Image.open(image_path) as img:
            img_w, img_h = img.size
    except FileNotFoundError:
        print(f"⚠️ Warning: Image not found: {image_path}")
        continue

    with open(label_path, "w") as f:
        for _, row in group.iterrows():
            # Convert class label to integer ID (default 0 if not numeric)
            class_id = int(row['label']) if str(row['label']).isdigit() else 0

            # Convert to YOLO normalized format
            x_center = (row['xmin'] + row['xmax']) / 2 / img_w
            y_center = (row['ymin'] + row['ymax']) / 2 / img_h
            width = (row['xmax'] - row['xmin']) / img_w
            height = (row['ymax'] - row['ymin']) / img_h

            f.write(f"{class_id} {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}\n")

print(f"✅ Pseudo-labels successfully written to {output_dir}")



✅ Loaded 302 detections
Index(['image_path', 'label', 'score', 'xmin', 'ymin', 'xmax', 'ymax'], dtype='object')
⚙️ Using dynamic CONF_THRESH = 0.258
➡️ After filtering: 226 detections kept (confidence ≥ 0.258)


Generating YOLO labels: 100%|████████████████████████████████████████████████████████| 217/217 [00:00<00:00, 994.41it/s]

✅ Pseudo-labels successfully written to /home/brhanu/thesis_project/data/labels/val


In [3]:
from ultralytics.utils.plotting import plot_results
plot_results('/home/brhanu/thesis_project/runs/detect/train_pseudo4/results.csv')


In [4]:
import os
import numpy as np

path = "/home/brhanu/thesis_project/data/labels/train"
for f in os.listdir(path)[:5]:
    arr = np.loadtxt(os.path.join(path,f))
    if arr.ndim == 1: arr = arr[None,:]
    print(f, arr.min(axis=0), arr.max(axis=0))



00000163_0.txt [          1     0.37254     0.29088     0.49303     0.30088] [          1     0.37254     0.29088     0.49303     0.30088]
00000060_0.txt [          0     0.94247     0.21468     0.28918     0.35558] [          0     0.94247     0.21468     0.28918     0.35558]
00000142_0.txt [          0     0.48606     0.89905     0.82075     0.67916] [          0     0.48606     0.89905     0.82075     0.67916]
00000041_0.txt [          0     0.15348    0.076509      0.2875     0.12296] [          0     0.15348    0.076509      0.2875     0.12296]
00000022_0.txt [          0     0.79347    0.079668     0.14138    0.074216] [          0     0.79347    0.079668     0.14138    0.074216]


In [5]:
import glob

label_paths = glob.glob("/home/brhanu/thesis_project/data/labels/**/*.txt", recursive=True)
for path in label_paths:
    with open(path) as f:
        lines = f.readlines()
    fixed = []
    for line in lines:
        cid, *coords = line.strip().split()
        if int(cid) < 2:
            fixed.append(line)
    with open(path, "w") as f:
        f.writelines(fixed)


In [6]:

import glob, os
train_imgs = glob.glob("/home/brhanu/thesis_project/data/images/train/*.jpg")
train_lbls = glob.glob("/home/brhanu/thesis_project/data/labels/train/*.txt")
print("Total images:", len(train_imgs))
print("Total labels:", len(train_lbls))
print("Images without labels:", len([i for i in train_imgs if not os.path.exists(i.replace("images", "labels").replace(".jpg", ".txt"))]))
print("Empty labels:", len([f for f in train_lbls if os.path.getsize(f)==0]))



Total images: 826
Total labels: 135
Images without labels: 693
Empty labels: 0


In [7]:
import pandas as pd
import matplotlib.pyplot as plt
import os

run_dir = "/home/brhanu/thesis_project/runs/detect/train_refine_long2"
results_csv = os.path.join(run_dir, "results.csv")

# Load YOLO results
df = pd.read_csv(results_csv)
print(df.head())

# Plot mAP, precision, recall
plt.figure(figsize=(10,5))
plt.plot(df['       metrics/mAP50(B)'], label='mAP@50')
plt.plot(df['       metrics/precision(B)'], label='Precision')
plt.plot(df['       metrics/recall(B)'], label='Recall')
plt.xlabel('Epoch')
plt.ylabel('Metric Value')
plt.title('YOLOv5 Training Metrics')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


   epoch      time  train/box_loss  train/cls_loss  train/dfl_loss  \
0      1   5.36976         2.63405         5.09573         2.45969   
1      2   9.00050         2.47112         4.96873         2.48844   
2      3  12.64750         2.63283         5.06958         2.54532   
3      4  16.55720         2.71771         5.13231         2.62366   
4      5  20.73190         2.52780         5.09163         2.48514   

   metrics/precision(B)  metrics/recall(B)  metrics/mAP50(B)  \
0                   0.0                0.0               0.0   
1                   0.0                0.0               0.0   
2                   0.0                0.0               0.0   
3                   0.0                0.0               0.0   
4                   0.0                0.0               0.0   

   metrics/mAP50-95(B)  val/box_loss  val/cls_loss  val/dfl_loss    lr/pg0  \
0                  0.0       2.79495       4.06246       2.82925  0.000327   
1                  0.0       2.78265  

KeyError: '       metrics/mAP50(B)'

In [8]:
import pandas as pd
import matplotlib.pyplot as plt
import os

run_dir = "/home/brhanu/thesis_project/runs/detect/train_refine_long3"
results_csv = os.path.join(run_dir, "results.csv")

df = pd.read_csv(results_csv)
print(df.head())

plt.figure(figsize=(10,5))
plt.plot(df['       metrics/mAP50(B)'], label='mAP@50')
plt.plot(df['       metrics/precision(B)'], label='Precision')
plt.plot(df['       metrics/recall(B)'], label='Recall')
plt.xlabel('Epoch')
plt.ylabel('Metric')
plt.title('YOLOv8 Training Metrics (train_refine_long3)')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig(os.path.join(run_dir, "training_metrics.png"), dpi=300)
plt.show()


   epoch      time  train/box_loss  train/cls_loss  train/dfl_loss  \
0      1   5.63611         2.44234         5.05953         2.43786   
1      2   9.56234         2.22079         4.86331         2.34290   
2      3  13.59530         2.39824         5.03638         2.43688   
3      4  17.54190         2.54611         5.03294         2.55687   
4      5  21.59430         2.32547         5.00915         2.38599   

   metrics/precision(B)  metrics/recall(B)  metrics/mAP50(B)  \
0               0.00000            0.00000           0.00000   
1               0.00000            0.00000           0.00000   
2               0.00410            0.00472           0.00330   
3               0.00207            0.00472           0.00119   
4               0.00231            0.00472           0.00120   

   metrics/mAP50-95(B)  val/box_loss  val/cls_loss  val/dfl_loss    lr/pg0  \
0              0.00000       2.58833       3.93229       2.77802  0.000327   
1              0.00000       2.63270  

KeyError: '       metrics/mAP50(B)'

In [11]:
import pandas as pd
import matplotlib.pyplot as plt
import os

# Path to YOLO training results directory
run_dir = "/home/brhanu/thesis_project/runs/detect/train_refine_long3"
results_csv = os.path.join(run_dir, "results.csv")

# Load YOLO training metrics
df = pd.read_csv(results_csv)
print("Loaded metrics columns:\n", df.columns.tolist())

# --- Plot 1: Performance Metrics ---
plt.figure(figsize=(10, 6))
plt.plot(df['metrics/mAP50(B)'], label='mAP@50')
plt.plot(df['metrics/precision(B)'], label='Precision')
plt.plot(df['metrics/recall(B)'], label='Recall')
plt.xlabel('Epoch')
plt.ylabel('Value')
plt.title('YOLOv8 Validation Metrics (train_refine_long3)')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig(os.path.join(run_dir, 'validation_metrics.png'), dpi=300)
plt.show()

# --- Plot 2: Training Loss Curves ---
plt.figure(figsize=(10, 6))
plt.plot(df['train/box_loss'], label='Box Loss')
plt.plot(df['train/cls_loss'], label='Class Loss')
plt.plot(df['train/dfl_loss'], label='DFL Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('YOLOv8 Training Loss Curves (train_refine_long3)')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig(os.path.join(run_dir, 'training_loss_curves.png'), dpi=300)
plt.show()

print(f"✅ Saved plots: validation_metrics.png and training_loss_curves.png in {run_dir}")

Loaded metrics columns:
 ['epoch', 'time', 'train/box_loss', 'train/cls_loss', 'train/dfl_loss', 'metrics/precision(B)', 'metrics/recall(B)', 'metrics/mAP50(B)', 'metrics/mAP50-95(B)', 'val/box_loss', 'val/cls_loss', 'val/dfl_loss', 'lr/pg0', 'lr/pg1', 'lr/pg2']


<Figure size 1000x500 with 0 Axes>

<Figure size 1000x500 with 0 Axes>

<Figure size 1000x600 with 0 Axes>

<Figure size 1000x600 with 1 Axes>

<Figure size 1000x600 with 1 Axes>

✅ Saved plots: validation_metrics.png and training_loss_curves.png in /home/brhanu/thesis_project/runs/detect/train_refine_long3


In [10]:
import pandas as pd
df = pd.read_csv("/home/brhanu/thesis_project/runs/detect/train_refine_long3/results.csv")
print(df.columns.tolist())


['epoch', 'time', 'train/box_loss', 'train/cls_loss', 'train/dfl_loss', 'metrics/precision(B)', 'metrics/recall(B)', 'metrics/mAP50(B)', 'metrics/mAP50-95(B)', 'val/box_loss', 'val/cls_loss', 'val/dfl_loss', 'lr/pg0', 'lr/pg1', 'lr/pg2']


In [1]:
import pandas as pd, os

csv_path = "/home/brhanu/thesis_project/results/vlm_structured_detections.csv"
out_dir = "/home/brhanu/thesis_project/data/labels/vlm_pseudo"
os.makedirs(out_dir, exist_ok=True)

# consistent class mapping
classes = {"animal":0,"horse":1,"family":2,"book":3,"house":4,"tree":5}

df = pd.read_csv(csv_path)
CONF_THRESH = 0.25
df = df[df["score"] >= CONF_THRESH]

for img, group in df.groupby("image_path"):
    img_name = os.path.splitext(os.path.basename(img))[0]
    txt = os.path.join(out_dir, f"{img_name}.txt")
    with open(txt,"w") as f:
        for _, r in group.iterrows():
            if r["label"] not in classes: continue
            cls = classes[r["label"]]
            W,H = 2048,2048
            x = ((r["xmin"]+r["xmax"])/2)/W
            y = ((r["ymin"]+r["ymax"])/2)/H
            w = (r["xmax"]-r["xmin"])/W
            h = (r["ymax"]-r["ymin"])/H
            f.write(f"{cls} {x:.6f} {y:.6f} {w:.6f} {h:.6f}\n")
print("✅ Pseudo-labels written to:", out_dir)


✅ Pseudo-labels written to: /home/brhanu/thesis_project/data/labels/vlm_pseudo


In [2]:
cp /home/brhanu/thesis_project/data/labels/vlm_pseudo/*.txt \
   /home/brhanu/thesis_project/data/labels/train/


SyntaxError: invalid syntax (627726758.py, line 1)